# HealthTrack Readmission Model + LangChain Alerts

In [ ]:
import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split

df = pd.read_csv('healthtrack_clean.csv')

feature_cols = ['age','length_of_stay','prev_admissions_12m','num_medications','department','discharge_disposition','insurance_type']
X = df[feature_cols].copy()
y = df['readmission_30d'].astype(int)
X_encoded = pd.get_dummies(X, columns=['department','discharge_disposition','insurance_type'], drop_first=False)

X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.2, random_state=42, stratify=y)
model = GradientBoostingClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]
print('Accuracy:', round(accuracy_score(y_test, y_pred), 4))
print('Precision:', round(precision_score(y_test, y_pred, zero_division=0), 4))
print('Recall:', round(recall_score(y_test, y_pred, zero_division=0), 4))
print('F1:', round(f1_score(y_test, y_pred, zero_division=0), 4))
print('ROC-AUC:', round(roc_auc_score(y_test, y_proba), 4))

# Score all 230 patients
all_proba = model.predict_proba(X_encoded)[:, 1]
df['readmission_probability'] = all_proba.round(4)
df['risk_tier'] = df['readmission_probability'].apply(lambda p: 'High' if p >= 0.65 else ('Medium' if p >= 0.40 else 'Low'))
df[['patient_id', 'readmission_probability', 'risk_tier']].head()

For this clinical workflow, recall is the most important metric because a false negative means missing a patient who is actually at high risk and may return without intervention. That miss can lead to avoidable harm, avoidable penalties, and missed care coordination opportunities. A false positive is less costly because it usually creates extra outreach work rather than missed care.

In [ ]:
import os

from langchain_core.prompts import PromptTemplate
from langchain_anthropic import ChatAnthropic

prompt = PromptTemplate(
    input_variables=['patient_id','age','department','length_of_stay','prev_admissions_12m','num_medications','discharge_disposition'],
    template=(
      'You are a hospital care coordinator. Write exactly 3 sentences in a clinical, non-alarmist tone. '
      'Sentence 1: state that the patient is High risk and name the top two contributing factors based on the provided fields. '
      'Sentence 2: recommend one specific follow-up action a care coordinator can execute immediately. '
      'Sentence 3: include a clear timeframe and coordination detail.\n\n'
      'patient_id: {patient_id}\nage: {age}\ndepartment: {department}\nlength_of_stay: {length_of_stay}\n'
      'prev_admissions_12m: {prev_admissions_12m}\nnum_medications: {num_medications}\ndischarge_disposition: {discharge_disposition}'
    )
)

if not os.getenv('ANTHROPIC_API_KEY'):
    raise EnvironmentError('Set ANTHROPIC_API_KEY before running this cell.')

llm = ChatAnthropic(model='claude-3-5-sonnet-20240620', temperature=0.2)
chain = prompt | llm

high_mask = df['risk_tier'] == 'High'
df['intervention_note'] = ''

for idx, row in df.loc[high_mask].iterrows():
    note = chain.invoke({
        'patient_id': row['patient_id'],
        'age': int(row['age']),
        'department': row['department'],
        'length_of_stay': int(row['length_of_stay']),
        'prev_admissions_12m': int(row['prev_admissions_12m']),
        'num_medications': int(row['num_medications']),
        'discharge_disposition': row['discharge_disposition'],
    })
    df.at[idx, 'intervention_note'] = note.content.strip()

df['alert_priority'] = df.apply(
    lambda r: 'URGENT' if (r['risk_tier'] == 'High' and r['prev_admissions_12m'] >= 2) else 'STANDARD', axis=1
)

alerts = df.loc[df['risk_tier'] == 'High', [
    'patient_id','age','department','risk_tier','readmission_probability','alert_priority','intervention_note'
]].copy()
alerts.to_csv('healthtrack_alerts_today.csv', index=False)
print(f'Saved {len(alerts)} high-risk alerts to healthtrack_alerts_today.csv')
alerts.head()